# 03 · Tensor-parallel MLP: forward and backward

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilesh-salpe/visual-ai/blob/main/build-it/distributed/03_tensor_parallel_training.ipynb) &nbsp; 📖 Concept deck: [Tensor and sequence parallelism](https://nilesh-salpe.github.io/visual-ai/distributed/04-tensor-sequence-parallelism/) &nbsp; ⏱️ Suggested time: **40 min**

**You will build:** a Megatron-style MLP split across t simulated GPUs (columns then rows), including the backward pass, with gradients that match the unsplit layer.

**How to use this notebook**
1. Run the **Setup** cell.
2. For each task, fill in the `TODO` in the code cell, then run the **Tests** cell under it. You'll see ⏳ (not done yet), ✅ (passed) or ❌ (failed, with the reason).
3. Stuck? The **Reference solutions** at the bottom are collapsed. Try first.

Everything runs on CPU with NumPy, so no GPU is needed.

In [ ]:
# @title Setup: run this first
import numpy as np, math, time, itertools, collections
np.set_printoptions(precision=4, suppress=True)

def check(name, test, impl):
    """Run `test(impl)` and report: ⏳ not implemented, ✅ passed, ❌ failed with the reason."""
    try:
        test(impl)
        print(f"✅ {name}: all tests passed")
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet. Fill in the TODO in the cell above, then re-run this cell.")
    except AssertionError as e:
        print(f"❌ {name}: {e}")


def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

r = np.random.default_rng(0)
T_, d, h = 6, 8, 16
X = r.normal(size=(T_, d)); A = r.normal(0, 0.3, (d, h)); Bw = r.normal(0, 0.3, (h, d)); dY = r.normal(size=(T_, d))

def mlp(X, A, B):
    return np.tanh(X @ A) @ B

def mlp_grads(X, A, B, dY):
    """Reference gradients of sum(Y * dY) for Y = tanh(XA) B."""
    Z = X @ A; H = np.tanh(Z)
    dB = H.T @ dY; dH = dY @ B.T; dZ = dH * (1 - H ** 2)
    return X.T @ dZ, dB, dZ @ A.T     # dA, dB, dX

## Task 1: Column-split A, row-split B

Megatron splits the first matrix **by columns** and the second **by rows**, so each GPU computes tanh on its own hidden units with no communication in between.
Return `(A_parts, B_parts)`, lists of length t, with `A_parts[i] = A[:, cols_i]` and `B_parts[i] = B[rows_i, :]` for the same index ranges.

In [ ]:
def split_weights(A, B, t):
    # TODO: use np.array_split on the hidden dimension
    raise NotImplementedError

In [ ]:
# @title Tests for Task 1: run after implementing
def _test_split_weights(f):
    Ap, Bp = f(A, Bw, 4)
    assert len(Ap) == 4 and all(a.shape == (d, h // 4) for a in Ap) and all(b.shape == (h // 4, d) for b in Bp)
    assert np.allclose(np.concatenate(Ap, 1), A) and np.allclose(np.concatenate(Bp, 0), Bw)

check("split_weights", _test_split_weights, split_weights)

## Task 2: Forward: one all-reduce

Each GPU i computes a **partial** output $Y_i = \tanh(X A_i) B_i$. The true output is the **sum** of the partials: one all-reduce. Return Y.

In [ ]:
def tp_forward(X, A_parts, B_parts):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 2: run after implementing
def _test_tp_forward(f):
    Ap, Bp = (split_weights_solution if 'split_weights_solution' in globals() else _split)(A, Bw, 4)
    assert np.allclose(f(X, Ap, Bp), mlp(X, A, Bw))

def _split(A, B, t):
    idx = np.array_split(np.arange(A.shape[1]), t); return [A[:, i] for i in idx], [B[i, :] for i in idx]

check("tp_forward", _test_tp_forward, tp_forward)

## Task 3: Backward: local weight gradients, one all-reduce for dX

With the incoming gradient dY (same on every GPU), each GPU i computes its weight gradients **locally**:
$H_i = \tanh(XA_i)$, $dB_i = H_i^\top dY$, $dZ_i = (dY B_i^\top)\odot(1 - H_i^2)$, $dA_i = X^\top dZ_i$.
The input gradient needs contributions from every GPU: $dX = \sum_i dZ_i A_i^\top$, which is the backward pass's all-reduce.
Return `(dA_parts, dB_parts, dX)`.

In [ ]:
def tp_backward(X, A_parts, B_parts, dY):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 3: run after implementing
def _test_tp_backward(f):
    Ap, Bp = _split(A, Bw, 4)
    dAp, dBp, dX = f(X, Ap, Bp, dY)
    rA, rB, rX = mlp_grads(X, A, Bw, dY)
    assert np.allclose(np.concatenate(dAp, 1), rA), "dA shards must concatenate to the full dA"
    assert np.allclose(np.concatenate(dBp, 0), rB), "dB shards must stack to the full dB"
    assert np.allclose(dX, rX), "dX needs the sum over GPUs (all-reduce)"

check("tp_backward", _test_tp_backward, tp_backward)

---
## Reference solutions
Try the tasks first. Each cell below is collapsed in Colab: double-click it (or click "Show code") to reveal the solution. Running it also checks the reference against the same tests.

In [ ]:
# @title Solution for Task 1: Column-split A, row-split B
def split_weights_solution(A, B, t):
    idx = np.array_split(np.arange(A.shape[1]), t)
    return [A[:, i] for i in idx], [B[i, :] for i in idx]

check("split_weights (reference)", _test_split_weights, split_weights_solution)

In [ ]:
# @title Solution for Task 2: Forward: one all-reduce
def tp_forward_solution(X, A_parts, B_parts):
    return sum(np.tanh(X @ a) @ b for a, b in zip(A_parts, B_parts))      # all-reduce = sum of partials

check("tp_forward (reference)", _test_tp_forward, tp_forward_solution)

In [ ]:
# @title Solution for Task 3: Backward: local weight gradients, one all-reduce for dX
def tp_backward_solution(X, A_parts, B_parts, dY):
    dA, dB, dX = [], [], 0
    for a, b in zip(A_parts, B_parts):
        H = np.tanh(X @ a)
        dB.append(H.T @ dY)
        dZ = (dY @ b.T) * (1 - H ** 2)
        dA.append(X.T @ dZ)
        dX = dX + dZ @ a.T                       # all-reduce in backward
    return dA, dB, dX

check("tp_backward (reference)", _test_tp_backward, tp_backward_solution)

---
## Experiments
**Communication per transformer layer.** Each block (attention + MLP) needs one all-reduce in the forward pass and one in the backward pass, each of size tokens × d. Sequence parallelism (Korthikanti et al.) replaces each all-reduce with a reduce-scatter + all-gather of the same total volume, but stores the LayerNorm/dropout activations sharded along the sequence instead of replicated. Below: per-layer activation memory with the paper's formula $sbh(34 + 5as/h)$ bytes, divided by t with tensor + sequence parallelism.

In [ ]:
s, b, hd, a = 2048, 1, 12288, 96     # GPT-3 175B-like: sequence, micro-batch, hidden, heads
base = s * b * hd * (34 + 5 * a * s / hd)
print(f"5as/h = {5 * a * s / hd:.0f} (the attention-score term; paper: 80 for GPT-3)")
for t in [1, 2, 4, 8]:
    print(f"t={t}: activations per layer {base / t / 2**30:6.2f} GiB with TP+SP; selective recompute keeps only 34·sbh/t = {34 * s * b * hd / t / 2**30:5.2f} GiB")

---
## Takeaways
* Column-then-row splitting lets each GPU apply the nonlinearity locally: one all-reduce forward and one backward per MLP (and per attention block).
* Weight gradients are local; only the activation gradient needs communication.
* Tensor parallelism communicates every layer, so it lives inside the NVLink domain; sequence parallelism also shards the activations TP alone would replicate.